# 32 · Comprobar las entradas de los modelos, sin entrenar

Selecciona **CPU** y ejecuta las celdas en orden. Solo se usa desarrollo: imágenes, edad, sexo e IMC, con las particiones ya aprobadas. La imputación y el escalado se calculan dentro del entrenamiento de cada pliegue. No se modifican originales ni se abre la prueba reservada.

Esta libreta instala un Python aislado y llama al código del repositorio en una revisión fija. Repite las comprobaciones de entrada y las compara con la referencia privada aprobada. La prueba de arquitectura usa **pesos aleatorios**, no descarga pesos preentrenados y no realiza entrenamiento ni evaluación predictiva.

Indica la misma carpeta privada de desarrollo que usaste en la libreta 31: contiene `transport.json`, los fragmentos y ahora `model_inputs_loading_reference_v1.json`. No selecciones la carpeta de prueba reservada.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, uuid

if os.environ.get('COLAB_GPU') == '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona CPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución.')
drive.mount('/content/drive')
REMOTE_PACKAGE_DIR = Path(os.environ.get('KNEE_DATA_ROOT') or input('Carpeta privada de desarrollo que contiene transport.json: ').strip()).absolute()
REFERENCE_SOURCE = Path(os.environ.get('KNEE_INPUT_REFERENCE_PATH', str(REMOTE_PACKAGE_DIR / 'model_inputs_loading_reference_v1.json'))).absolute()
for item in (REMOTE_PACKAGE_DIR, REFERENCE_SOURCE):
    if any(p.is_symlink() or getattr(p, 'is_junction', lambda: False)() for p in (item, *item.parents)):
        raise ValueError('No se permiten enlaces en las rutas privadas.')
if not REMOTE_PACKAGE_DIR.is_dir() or not REFERENCE_SOURCE.is_file() or not 0 < REFERENCE_SOURCE.stat().st_size <= 4 * 1024**2:
    raise ValueError('No se encontró la carpeta de desarrollo o la referencia aprobada.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_input_check_'))
REPO_DIR = SESSION_ROOT / 'repository'
CODE_REVISION = 'b88e5229e51d33b264ff5ae0464ddabc337dc578'

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        print(result.stdout[-12000:])
        print(result.stderr[-12000:])
        raise RuntimeError('La comprobación se detuvo; no se autoriza entrenamiento.')
    return result.stdout

def file_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

checked_run(['git', 'clone', '--quiet', '--config', 'core.autocrlf=false', '--filter=blob:none', '--no-checkout', 'https://github.com/AlonsoYess/Knee.git', str(REPO_DIR)])
checked_run(['git', 'checkout', '--quiet', '--detach', CODE_REVISION], cwd=REPO_DIR)
if checked_run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).strip() != CODE_REVISION:
    raise RuntimeError('La revisión del código no coincide con la aprobada.')
print('Código del repositorio comprobado. Aún no se han abierto imágenes.')

## Preparar el entorno CPU aislado

La primera ejecución descarga el intérprete y las bibliotecas fijadas, no modelos preentrenados. No cambia las bibliotecas del Python principal de Colab.

In [ ]:
print('Preparando el intérprete aislado y las bibliotecas CPU…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/setup_model_inputs_cpu.py'), '--destination', str(SESSION_ROOT / 'cpu_runtime'), '--repository-root', str(REPO_DIR)]))
NEURAL_PYTHON = Path(runtime['python_path'])
if not NEURAL_PYTHON.is_file():
    raise RuntimeError('No se encontró el intérprete CPU preparado.')
RUN_ENV = dict(os.environ)
for key in ('PYTHONHOME', 'PYTHONPATH', 'VIRTUAL_ENV'):
    RUN_ENV.pop(key, None)
RUN_ENV.update(PYTHONNOUSERSITE='1', PYTHONDONTWRITEBYTECODE='1', PYTHONPATH=str(REPO_DIR / 'src') + os.pathsep + str(REPO_DIR / 'tests'), HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
print('Entorno CPU preparado. No se instalaron pesos preentrenados.')

In [ ]:
if 'knee' in sys.modules:
    raise RuntimeError('Reinicia el entorno y ejecuta las celdas en orden para evitar código cargado anteriormente.')
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee.colab_package_transport import stage_development_package
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
LOCAL_PACKAGE_DIR.mkdir()
transport = stage_development_package(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR)
if (LOCAL_PACKAGE_DIR / 'reserved_test_package.zip').exists():
    raise RuntimeError('La prueba reservada no debe estar en el almacenamiento de esta comprobación.')
REFERENCE_HASH = file_sha(REFERENCE_SOURCE)
LOCAL_REFERENCE = SESSION_ROOT / 'approved_reference.json'
with REFERENCE_SOURCE.open('rb') as source, LOCAL_REFERENCE.open('xb') as target:
    shutil.copyfileobj(source, target, length=1024**2)
if file_sha(LOCAL_REFERENCE) != REFERENCE_HASH or file_sha(REFERENCE_SOURCE) != REFERENCE_HASH:
    raise RuntimeError('La referencia cambió durante su copia.')
print('Solo desarrollo fue reconstruido y verificado en almacenamiento temporal. Los archivos de Drive no se modificaron.')

In [ ]:
TEST_MODULES = ['test_clinical_fold_inputs', 'test_model_input_adapters', 'test_model_inputs_preflight', 'test_colab_package_transport', 'test_model_inputs_colab', 'test_model_inputs_cpu_setup']
EXPECTED_TESTS = 108
print('Ejecutando las pruebas de preparación, aislamiento y carga…')
test_process = subprocess.run([str(NEURAL_PYTHON), '-B', '-m', 'unittest', *TEST_MODULES], cwd=REPO_DIR, env=RUN_ENV, capture_output=True, text=True)
test_log = test_process.stdout + test_process.stderr
counts = re.findall(r'Ran (\d+) tests', test_log)
if test_process.returncode != 0 or counts != [str(EXPECTED_TESTS)] or re.search(r'skipped=|FAILED|ERROR', test_log):
    print(test_log[-16000:])
    raise RuntimeError('Las pruebas de esta libreta no fueron completas; no continúes.')
CPU_TESTS = {'passed': True, 'tests_run': EXPECTED_TESTS, 'skipped': 0, 'scope': TEST_MODULES}
print(f'Pruebas aprobadas: {EXPECTED_TESTS}, ninguna omitida.')

## Comprobar los cinco pliegues

Se vuelven a calcular los estados clínicos y a recorrer todas las entradas de entrenamiento y validación. Se exige coincidencia exacta con la referencia de los datos y del procesamiento clínico. Las arquitecturas solo hacen una pasada de compatibilidad, sin gradientes ni actualización de pesos.

In [ ]:
LOCAL_OUTPUT = SESSION_ROOT / 'model_inputs_check'
EXECUTION_ENVIRONMENT = 'local_rehearsal' if globals().get('_LOCAL_REHEARSAL') is True else 'google_colab_cpu'
print('Comprobando todas las entradas y la compatibilidad CPU, sin entrenar…')
loading_log = checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/check_model_inputs_colab.py'), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-record', str(LOCAL_REFERENCE), '--repository-root', str(REPO_DIR), '--output-dir', str(LOCAL_OUTPUT), '--execution-environment', EXECUTION_ENVIRONMENT], cwd=REPO_DIR, env=RUN_ENV)
print(loading_log)
loading = json.loads((LOCAL_OUTPUT / 'closure_summary.json').read_text(encoding='utf-8'))
if loading['status'] != 'development_inputs_and_cpu_loading_verified' or loading['reference_matches_training_only_states_and_all_loader_hashes'] is not True or loading['training_executed'] is not False or loading['reserved_test_opened'] is not False:
    raise RuntimeError('No se obtuvo una comprobación completa sin entrenamiento.')
if checked_run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).strip() != CODE_REVISION or file_sha(REFERENCE_SOURCE) != REFERENCE_HASH:
    raise RuntimeError('El código o la referencia cambiaron durante la comprobación.')

In [ ]:
report_root = SESSION_ROOT if EXECUTION_ENVIRONMENT == 'local_rehearsal' else REMOTE_PACKAGE_DIR
FINAL_OUTPUT_DIR = report_root / ('model_input_check_' + uuid.uuid4().hex)
FINAL_OUTPUT_DIR.mkdir(exist_ok=False)
for source in sorted(LOCAL_OUTPUT.rglob('*')):
    if source.is_symlink():
        raise RuntimeError('Una salida contiene un enlace no permitido.')
    if source.is_file():
        target = FINAL_OUTPUT_DIR / source.relative_to(LOCAL_OUTPUT)
        target.parent.mkdir(parents=True, exist_ok=True)
        with source.open('rb') as incoming, target.open('xb') as destination:
            shutil.copyfileobj(incoming, destination, length=1024**2)
        if file_sha(source) != file_sha(target):
            raise RuntimeError('Una salida guardada no coincide con el resultado local.')
FINAL_CHECK = {
    'version': loading['version'], 'status': 'passed',
    'colab_runtime_executed': loading['colab_runtime_executed'],
    'local_rehearsal': loading['local_rehearsal'], 'cpu_only_guard_passed': True,
    'code_revision': CODE_REVISION, 'cpu_tests': CPU_TESTS,
    'development_participants': loading['development_participants'],
    'development_knees': loading['development_knees'],
    'clinical_folds_verified': loading['clinical_states_fitted'],
    'loader_row_visits_verified': loading['total_loader_row_visits'],
    'reference_matches_training_only_states_and_all_loader_hashes': True,
    'clinical_preprocessing_fitted_inside_training_folds_only': True,
    'model_specific_normalization_applied': True,
    'image_geometry_unchanged': True, 'architecture_probe_weights': 'random_weights_none',
    'pretrained_weights_loaded': False, 'training_executed': False,
    'originals_modified': False, 'partitions_modified': False,
    'reserved_test_predictive_evaluation_opened': False,
    'result_files_saved_and_readback_verified': True
}
FINAL_REPORT_PATH = FINAL_OUTPUT_DIR / 'colab_model_inputs_check.json'
with FINAL_REPORT_PATH.open('x', encoding='utf-8') as stream:
    json.dump(FINAL_CHECK, stream, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False)
    stream.write('\n')
if json.loads(FINAL_REPORT_PATH.read_text(encoding='utf-8')) != FINAL_CHECK:
    raise RuntimeError('La lectura del registro final no coincide.')
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('Comprobación terminada y guardada. No se entrenó ni se abrió la prueba reservada.')

Comparte únicamente el **JSON final**. Los resultados clínicos y el registro detallado quedan en una carpeta nueva junto al paquete de desarrollo. Ninguna salida anterior se reemplaza. Un ensayo local se identifica como tal y nunca escribe resultados en Drive. No necesitas ejecutar las libretas anteriores, abrir la prueba reservada ni enviar imágenes.